# Module 04-1: 문서 요약 에이전트 구현 (솔루션)

## 학습 목표

1. **State 설계**: 7개 필드로 구성된 `DocumentSummaryState`를 TypedDict로 정의한다
2. **4노드 파이프라인**: fetch_document → analyze_content → generate_summary → format_output 흐름을 구현한다
3. **조건부 에러 분기**: `_route_after_fetch`, `_route_after_analyze`로 에러 발생 시 조기 종료한다
4. **functools.partial 의존성 주입**: LLM을 노드 함수에 외부에서 주입하여 테스트 가능하게 만든다
5. **FakeLLM E2E 테스트**: 실제 API 없이 전체 파이프라인을 테스트한다

---
> 참고 문서: `docs/part2-first-agent/04-building-first-agent.md`

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import json
from typing import TypedDict
from functools import partial
from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: DocumentSummaryState 정의 (솔루션)

In [ ]:
# 솔루션
class DocumentSummaryState(TypedDict):
    source_text: str
    document: dict | None
    analysis: dict | None
    summary: str | None
    final_output: str | None
    current_step: str
    error: str | None

print(f"필드 목록: {list(DocumentSummaryState.__annotations__.keys())}")

In [ ]:
# 검증 셀
assert 'DocumentSummaryState' in dir(), "DocumentSummaryState가 정의되어야 합니다"
fields = list(DocumentSummaryState.__annotations__.keys())
required_fields = ['source_text', 'document', 'analysis', 'summary', 'final_output', 'current_step', 'error']
for f in required_fields:
    assert f in fields, f"'{f}' 필드가 없습니다"
print("✅ 실습 1 완료!")

## 실습 2: fetch_document 노드 구현 (솔루션)

In [ ]:
# 솔루션
def fetch_document(state: DocumentSummaryState) -> dict:
    source_text = state["source_text"]
    if not source_text or not source_text.strip():
        return {"error": "문서가 비어있습니다. 텍스트를 입력해주세요.", "current_step": "error"}
    lines = source_text.strip().split("\n")
    title = lines[0] if lines else "제목 없음"
    content = "\n".join(lines[1:]).strip() if len(lines) > 1 else source_text
    word_count = len(source_text.split())
    document = {"title": title, "content": content, "word_count": word_count, "line_count": len(lines)}
    print(f"  [fetch_document] 문서 로드 완료: '{title}' ({word_count}단어)")
    return {"document": document, "current_step": "fetched"}


print("fetch_document 구현 완료")

In [ ]:
# 검증 셀
base_state = {"source_text": "", "document": None, "analysis": None,
              "summary": None, "final_output": None, "current_step": "start", "error": None}

r1 = fetch_document({**base_state, "source_text": "Python 가이드\n파이썬은 간결한 언어입니다."})
assert r1["current_step"] == "fetched"
assert r1["document"]["title"] == "Python 가이드"

r2 = fetch_document({**base_state, "source_text": ""})
assert r2.get("error") is not None
assert r2["current_step"] == "error"
print("✅ 실습 2 완료!")

## 실습 3: analyze_content 노드 구현 (솔루션)

In [ ]:
# 솔루션
def analyze_content(state: DocumentSummaryState, llm=None) -> dict:
    document = state.get("document")
    if not document:
        return {"error": "문서가 로드되지 않았습니다.", "current_step": "error"}
    if llm is None:
        return {"error": "LLM이 설정되지 않았습니다.", "current_step": "error"}
    prompt = (
        f"다음 문서를 분석해주세요.\n\n"
        f"제목: {document['title']}\n"
        f"내용: {document['content']}\n\n"
        f'JSON 형식으로 응답해주세요: {{"keywords": [...], "topic": "...", "difficulty": "초급|중급|고급"}}'
    )
    try:
        response = llm.invoke(prompt)
        content = response.content
        try:
            analysis = json.loads(content)
        except json.JSONDecodeError:
            analysis = {"keywords": [], "topic": content[:100], "difficulty": "중급", "raw_response": content}
        print(f"  [analyze_content] 분석 완료: 주제='{analysis.get('topic', 'N/A')}'")
        return {"analysis": analysis, "current_step": "analyzed"}
    except Exception as e:
        return {"error": f"분석 중 오류 발생: {str(e)}", "current_step": "error"}


print("analyze_content 구현 완료")

In [ ]:
# 검증 셀
fake_llm_test = FakeLLM(
    responses={"분석": json.dumps({"keywords": ["Python"], "topic": "웹 개발", "difficulty": "중급"}, ensure_ascii=False)},
    default_response="분석 완료"
)
base_state = {"source_text": "", "document": {"title": "Python 가이드", "content": "분석 대상 텍스트"},
              "analysis": None, "summary": None, "final_output": None, "current_step": "fetched", "error": None}

r = partial(analyze_content, llm=fake_llm_test)(base_state)
assert r["current_step"] == "analyzed"
assert r["analysis"] is not None

r2 = analyze_content(base_state, llm=None)
assert r2.get("error") is not None
print("✅ 실습 3 완료!")

## 실습 4: generate_summary + format_output 노드 구현 (솔루션)

In [ ]:
# 솔루션
def generate_summary(state: DocumentSummaryState, llm=None) -> dict:
    if llm is None:
        return {"error": "LLM이 설정되지 않았습니다.", "current_step": "error"}
    document = state.get("document", {})
    analysis = state.get("analysis", {})
    prompt = (
        f"다음 문서를 요약해주세요.\n\n"
        f"제목: {document.get('title', 'N/A')}\n"
        f"내용: {document.get('content', 'N/A')}\n"
        f"분석 결과: {json.dumps(analysis, ensure_ascii=False)}\n\n"
        f"3줄 이내로 요약해주세요."
    )
    try:
        response = llm.invoke(prompt)
        summary = response.content
        print(f"  [generate_summary] 요약 생성 완료 ({len(summary)}자)")
        return {"summary": summary, "current_step": "summarized"}
    except Exception as e:
        return {"error": f"요약 생성 중 오류 발생: {str(e)}", "current_step": "error"}


def format_output(state: DocumentSummaryState) -> dict:
    document = state.get("document", {})
    analysis = state.get("analysis", {})
    summary = state.get("summary", "요약 없음")
    keywords = analysis.get("keywords", [])
    keywords_str = ", ".join(keywords) if keywords else "추출된 키워드 없음"
    output_parts = [
        "=" * 50,
        "  문서 요약 결과",
        "=" * 50,
        "",
        f"  제목: {document.get('title', 'N/A')}",
        f"  단어 수: {document.get('word_count', 'N/A')}",
        f"  주제: {analysis.get('topic', 'N/A')}",
        f"  난이도: {analysis.get('difficulty', 'N/A')}",
        f"  키워드: {keywords_str}",
        "",
        "-" * 50,
        "  요약:",
        f"  {summary}",
        "-" * 50,
    ]
    final_output = "\n".join(output_parts)
    print("  [format_output] 포맷팅 완료")
    return {"final_output": final_output, "current_step": "completed"}


print("generate_summary, format_output 구현 완료")

In [ ]:
# 검증 셀
fake_llm2 = FakeLLM(responses={"요약": "FastAPI는 고성능 Python 웹 프레임워크입니다."})
test_state = {
    "source_text": "FastAPI 시작하기",
    "document": {"title": "FastAPI 시작하기", "content": "FastAPI는 Python 웹 프레임워크입니다.", "word_count": 6, "line_count": 1},
    "analysis": {"keywords": ["Python", "FastAPI"], "topic": "웹 개발", "difficulty": "중급"},
    "summary": None, "final_output": None, "current_step": "analyzed", "error": None
}

r_sum = partial(generate_summary, llm=fake_llm2)(test_state)
assert r_sum["current_step"] == "summarized"
assert r_sum["summary"] is not None

test_state["summary"] = r_sum["summary"]
r_fmt = format_output(test_state)
assert r_fmt["current_step"] == "completed"
assert r_fmt["final_output"] is not None
print("✅ 실습 4 완료!")

## 실습 5: 조건부 분기 함수 및 그래프 조립 (솔루션)

In [ ]:
# 솔루션
def _route_after_fetch(state: DocumentSummaryState) -> str:
    return "error" if state.get("error") else "analyze"


def _route_after_analyze(state: DocumentSummaryState) -> str:
    return "error" if state.get("error") else "summarize"


def build_summarizer_graph(llm=None):
    graph = StateGraph(DocumentSummaryState)
    graph.add_node("fetch_document", fetch_document)
    graph.add_node("analyze_content", partial(analyze_content, llm=llm))
    graph.add_node("generate_summary", partial(generate_summary, llm=llm))
    graph.add_node("format_output", format_output)
    graph.set_entry_point("fetch_document")
    graph.add_conditional_edges(
        "fetch_document", _route_after_fetch,
        {"analyze": "analyze_content", "error": END}
    )
    graph.add_conditional_edges(
        "analyze_content", _route_after_analyze,
        {"summarize": "generate_summary", "error": END}
    )
    graph.add_edge("generate_summary", "format_output")
    graph.add_edge("format_output", END)
    return graph.compile()


print("그래프 조립 함수 정의 완료")

In [ ]:
# 검증 셀
s_ok = {"source_text": "x", "document": None, "analysis": None, "summary": None,
        "final_output": None, "current_step": "fetched", "error": None}
s_err = {**s_ok, "error": "에러 발생"}

assert _route_after_fetch(s_ok) == "analyze"
assert _route_after_fetch(s_err) == "error"
assert _route_after_analyze(s_ok) == "summarize"
assert _route_after_analyze(s_err) == "error"
print("✅ 분기 함수 검증 완료!")

## 실습 6: FakeLLM E2E 테스트 (솔루션)

In [ ]:
# 솔루션
fake_llm = FakeLLM(
    responses={
        "분석": json.dumps({
            "keywords": ["Python", "FastAPI", "웹 프레임워크"],
            "topic": "Python 웹 개발",
            "difficulty": "중급",
        }, ensure_ascii=False),
        "요약": (
            "본 문서는 FastAPI를 활용한 Python 웹 개발 방법을 설명합니다. "
            "REST API 설계 원칙과 비동기 처리 패턴을 중심으로 다룹니다."
        ),
    },
    default_response="요청을 처리했습니다.",
)

graph = build_summarizer_graph(llm=fake_llm)

initial_state = {
    "source_text": (
        "FastAPI 시작하기\n"
        "FastAPI는 Python 3.7+로 API를 구축하기 위한 고성능 웹 프레임워크입니다."
    ),
    "document": None, "analysis": None, "summary": None,
    "final_output": None, "current_step": "start", "error": None,
}

error_state = {
    "source_text": "",
    "document": None, "analysis": None, "summary": None,
    "final_output": None, "current_step": "start", "error": None,
}

result_normal = graph.invoke(initial_state)
result_error = graph.invoke(error_state)

print(f"정상 경로 - current_step: {result_normal.get('current_step')}")
print(f"에러 경로 - current_step: {result_error.get('current_step')}, error: {result_error.get('error')}")

In [ ]:
# E2E 검증 셀
assert result_normal.get("final_output") is not None, "정상 경로에서 final_output이 설정되어야 합니다"
assert result_normal.get("current_step") == "completed"
assert result_normal.get("error") is None

assert result_error.get("error") is not None
assert result_error.get("current_step") == "error"
assert result_error.get("final_output") is None

print("✅ 실습 6 완료! E2E 테스트 성공")

## 정리

### 오늘 배운 내용
- **7필드 State 설계**: source_text → document → analysis → summary → final_output + current_step + error
- **4노드 파이프라인**: fetch → analyze → summarize → format
- **조건부 에러 분기**: `add_conditional_edges`로 에러 시 조기 종료
- **functools.partial**: LLM을 외부에서 주입하여 테스트 가능한 아키텍처 구현
- **FakeLLM E2E 테스트**: 실제 API 없이 전체 파이프라인 검증